In [ ]:
import pandas as pd
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse, unquote
from tqdm import tqdm
import re
from IPython.display import display
import ipaddress

tqdm.pandas()

# two files, subscribed and searched, the searched urls are not very clean and may contain benign urls (as reference or context)
# we'll use just urls from subscribed pulses since they are much cleaner and we already have enough malicious urls
alienVault_path = '../../datasets/malicious/alienVault/otx_urls_subscribed.txt'
# 22k url, missing scheme, one column, no header
# we won't use the entire urls, we'll just use the resolved ones since the original ones don't have schemes
cybercrimeTracker_path = '../../datasets/malicious/cybercrimeTracker/resolved.csv'
# 784k url, use as is, one column, no header
github_phishing_database_path = '../../datasets/malicious/github/Phishing-Database/ALL-phishing-links.lst'
# folder path, contains three folders, each contains files with urls, ips...
# go through all the csv files, read them with comment='#' and no header, filter urls
maltrial_path = '../../datasets/malicious/maltrail/all.csv'
# 53k url, take urls with label=0, many columns, use the 'url' column
phishTank_path = '../../datasets/malicious/phishTank/phishTank.csv'
# 235k url, use as is, many columns, use the 'URL' column
phiUSIIL_path = '../../datasets/malicious/PhiUSIIL-phishing-urls/PhiUSIIL_Phishing_URL_Dataset.csv'
# sophoslabs folder path, contains many csv files, with domains, ips...
# go through all the csv files and extract urls, also clean them (ex: https[//:]... => https//:...)
sophoslabs_path = '../../datasets/malicious/sophoslabs/all.csv'
# use threatfox urls as is and extract the urls from the full file
threatfox_urls_path = '../../datasets/malicious/threatfox/full_urls.csv'
threatfox_full_path = '../../datasets/malicious/threatfox/full.csv'
# 82k url, use as is, many columns, use the 'url' column
urlhaus_path = '../../datasets/malicious/URLhaus/urls.csv'

In [3]:
alienVault_df = pd.read_csv(alienVault_path, header=None, names=['url'], sep='\t')

alienVault_df

,url
0,http://194.36.191.186/vycvebnbesre4q/uqcwvebnl...
1,http://ncdnncjnrjmainmain.us/new/new/home/defa...
2,http://92.63.197.153/www/2.exe
3,https://support.ms-live.us/update/02583235891M...
4,https://mairie-fimela.com/UE.php
...,...
38804,http://5.104.105.194/adminxx5xx/i/p.php
38805,https://we.tl/t-cFvm5QQlyV
38806,https://www.dawnpk.org/news-fce44fe5
38807,http://zhnapny.biz:2177


In [4]:
cybercrimeTracker_df = pd.read_csv(cybercrimeTracker_path, index_col=0)

cybercrimeTracker_df

,url
0,https://5.182.86.32/auth/login
1,https://79.137.194.188/auth/login
2,https://achillharpfestival.ie/wp-content/plugi...
3,https://achillharpfestival.ie/wp-content/plugi...
4,http://unwaveringmedia.com/wp-content/plugins/...
...,...
3672,https://vxml4.delacon.com.au/blog/
3673,https://moneybooster.info/login.php
3674,https://agilebasecamp.org/back/adm/index.php?m...
3675,https://ibicinemas.com.br/inc/rm.php


In [5]:
github_phishing_database_df = pd.read_csv(github_phishing_database_path, header=None, names=['url'], sep='\t')

github_phishing_database_df

,url
0,ftp://188.128.111.33/IPTV/TV1324/view.html
1,ftp://188.128.111.33/web/sec.htm
2,ftp://redacted@redacted.invalid:redacted@redac...
3,http://000000000000000000000000000000000000000...
4,http://00000000000000000000000000000000000000d...
...,...
784292,http://zylon-hub.pages.dev
784293,http://zymylyydkw.duckdns.org/en
784294,http://zzarhxmjlu.blogspot.com
784295,http://zzarhxmjlu.blogspot.tw


In [6]:
maltrial_df = pd.read_csv(maltrial_path, index_col=0)

maltrial_df

,url
0,aps.kemoge.net
1,taosha.cc
2,123.194.235.37:49320
3,1.34.220.200:52672
4,186.188.229.46:44977
...,...
1324350,dauled.com
1324351,erbolsan.com
1324352,kasym500.com
1324353,kasymdev.com


In [7]:
phishTank_df = pd.read_csv(phishTank_path)[['url']]

phishTank_df

,url
0,https://schiwezr-sbb-ccfd78.info/3/cff-sbb-ffs...
1,https://innavors.com/p/256713726
2,https://a4r6m9.lat/efivwbnu/wFAtZc
3,https://ritva.eclecticdigital.in/app/Providers...
4,http://cryptoconsultlineinfo.netlify.app
...,...
53148,http://webmailadmin0.myfreesites.net/
53149,http://www.formbuddy.com/cgi-bin/formdisp.pl?u...
53150,http://www.formbuddy.com/cgi-bin/formdisp.pl?u...
53151,http://www.habbocreditosparati.blogspot.com/


In [8]:
phiUSIIL_df = pd.read_csv(phiUSIIL_path)
phiUSIIL_df = phiUSIIL_df.loc[phiUSIIL_df['label'] == 0, ['URL']].rename(columns={'URL': 'url'})

phiUSIIL_df

,url
11,http://www.teramill.com
20,http://www.f0519141.xsph.ru
21,http://www.shprakserf.gq
27,https://service-mitld.firebaseapp.com/
28,http://www.kuradox92.lima-city.de
...,...
235780,https://ww.prestamo.enlinea.pe.vpphoangha.vn/
235782,http://goldenrod-motley-texture.glitch.me/hvwa...
235783,https://bancolombia.com1home0892.repl.co/?2
235784,https://aol-108318.weeblysite.com/


In [9]:
# for both maltrial and sophoslabs urls you can try to resolve some incomplete urls the way we did before if you need more malicious urls
sophoslabs_df = pd.read_csv(sophoslabs_path, index_col=0).dropna(subset=['indicator_type', 'data'])

sophoslabs_df = sophoslabs_df[sophoslabs_df['indicator_type'].str.lower().isin(['url', 'domain'])][['data']].rename(columns={'data': 'url'})

sophoslabs_df

,url
1,abdurantom.com
2,airwiseq.cf
3,appconnect.bekapro.xyz
4,apps-notification.com
5,goodthings.ml
...,...
17947,albanallahacrab.club
17948,dclogictrust.com
17949,masskwearing.cyou
17950,padishahmurrka.best


In [10]:
threatfox_urls_df = pd.read_csv(threatfox_urls_path, skipinitialspace=True)[['ioc_value']].rename(columns={'ioc_value': 'url'})

threatfox_urls_df

,url
0,https://logicmesh.pro/api/bot/heartbeat
1,http://185.174.133.12/98926703060a4fbf.php
2,https://dinglev.cyou/api
3,https://74.0.48.145/
4,http://165.232.165.152:8080/xoner.sh
...,...
13301,http://51.195.53.221/p.php/cfOoZYb0LXPms
13302,http://kbfvzoboss.bid/alien/fre.php
13303,http://alphastand.trade/alien/fre.php
13304,http://alphastand.win/alien/fre.php


In [11]:
threatfox_full_df = pd.read_csv(threatfox_full_path, skipinitialspace=True)
threatfox_full_df = threatfox_full_df.loc[threatfox_full_df['ioc_type'] == 'url', ['ioc_value']].rename(columns={'ioc_value': 'url'})

threatfox_full_df

,url
5,http://91.92.243.29/bnoda
21,https://spk.megaexdistribuidora.com.br/
22,https://spk.emiraride.com/
36,https://oly.megaexdistribuidora.com.br/
37,https://74.0.48.35/
...,...
188913,http://51.195.53.221/p.php/cfOoZYb0LXPms
188978,http://kbfvzoboss.bid/alien/fre.php
188979,http://alphastand.trade/alien/fre.php
188980,http://alphastand.win/alien/fre.php


In [12]:
urlhaus_df = pd.read_csv(urlhaus_path)[['url']]

urlhaus_df

,url
0,http://110.37.0.170:60238/bin.sh
1,http://221.15.193.178:33431/i
2,http://158.94.210.195/pay
3,http://158.94.210.195/bin
4,http://158.94.210.195/realtek
...,...
81995,http://asl-company.ru/Notification-de-facture-...
81996,http://xn--90abegbttpjb3bzb2j.xn--p1ai/doc/En/...
81997,http://kuzina-teatr.ru/newsletter/US_us/FILE/I...
81998,http://robertrowe.com/DOC/Past-Due-invoice/


In [113]:
df = pd.concat([
		alienVault_df, cybercrimeTracker_df, github_phishing_database_df,
		maltrial_df, phishTank_df, phiUSIIL_df, sophoslabs_df,
		threatfox_urls_df, threatfox_full_df, urlhaus_df
  ])

df

,url
0,http://194.36.191.186/vycvebnbesre4q/uqcwvebnl...
1,http://ncdnncjnrjmainmain.us/new/new/home/defa...
2,http://92.63.197.153/www/2.exe
3,https://support.ms-live.us/update/02583235891M...
4,https://mairie-fimela.com/UE.php
...,...
81995,http://asl-company.ru/Notification-de-facture-...
81996,http://xn--90abegbttpjb3bzb2j.xn--p1ai/doc/En/...
81997,http://kuzina-teatr.ru/newsletter/US_us/FILE/I...
81998,http://robertrowe.com/DOC/Past-Due-invoice/


In [114]:
print(df.isna().sum())
df.info()
df.describe()

url    0
dtype: int64
<class 'pandas.DataFrame'>
Index: 2421275 entries, 0 to 81999
Data columns (total 1 columns):
 #   Column  Dtype
---  ------  -----
 0   url     str  
dtypes: str(1)
memory usage: 122.6 MB


,url
count,2421275
unique,2368706
top,
freq,46


In [ ]:
def clean_url(url):
	if len(url) > 2000:
		return None
	url = url.replace('[.]', '.')\
		.replace('[:]', ':')\
		.replace('[://]', '://')\
		.replace('[dot]', '.')\
		.replace('[..]', '..')
  
	url = url.rstrip("'\"/")
 
  patterns = {
    r"h\[xx\]p": "http",
    r"hxxp": "http",
    r"htps": "https",
    r"htttp": "http",
	}

  for pattern, replacement in patterns.items():
    url = re.sub(pattern, replacement, url, flags=re.IGNORECASE)

	other_email_patterns = [
		r'email=redacted%40redacted.invalid',
		r'email=3mail%40b.c',
		r'email=a%40a.c',
		r'email=redacted_email',
		r'\[x@x.com\]'
	]

	for pattern in other_email_patterns:
		url = re.sub(pattern, 'email=example123@gmail.com', url, flags=re.IGNORECASE)

	# Email placeholder patterns - substitute with example email
	email_patterns = [
   	r'%redacted@redacted.invalid',
		r'\[email[a-z0-9%]*protected\]',
		r'\[-?email-?\]?',
	]
	
	for pattern in email_patterns:
		url = re.sub(pattern, 'johndoe@gmail.com', url, flags=re.IGNORECASE)
 
	# Parse URL to work with components
	try:
		parsed = urlparse(url)
	except:
		return None
	
	# Must have scheme and netloc
	if not parsed.scheme or not parsed.netloc:
		return None
	
	# Filter out if path contains unresolvable patterns
	unresolvable_path_patterns = [
		r'\[MRWEEBEE\]',
		r'\[Root-Mrx\.Tk\]',
		r'\[B~7\]qxt5jEp',
		r'\[223ail_',
		r'\[RandStr,',
		r'\[backup-site-antigo\]',
	]
	for pattern in unresolvable_path_patterns:
		if re.search(pattern, parsed.path, re.IGNORECASE):
			return None
				
	# Redacted patterns to strip from components
	redacted_patterns = [
		r'\[redacted[a-z0-9_]*\]',
		r'\[truncated[a-z0-9_]*\]',
		r'\[some_font_package\]',
		r'\[32-characters\]',
		r'\[private\]',
		r'\[fin\]',
		r'\[fud\]',
		r'\[upg\]',
		r'\[simple%20info\]',
		r'\[arch\]',
		r'\[custom_data\]',
	]
	
	# Check netloc - filter out entire URL if redacted
	for pattern in redacted_patterns:
		if re.search(pattern, parsed.netloc, re.IGNORECASE):
			return None
	
	# Clean path by removing redacted patterns
	cleaned_path = parsed.path
	for pattern in redacted_patterns:
		cleaned_path = re.sub(pattern, '', cleaned_path, flags=re.IGNORECASE)
	# Remove potential double slashes after removal
	cleaned_path = re.sub(r'/+', '/', cleaned_path)
	parsed = parsed._replace(path=cleaned_path)
	
	# Clean params by removing redacted patterns
	cleaned_params_str = parsed.params
	for pattern in redacted_patterns:
		cleaned_params_str = re.sub(pattern, '', cleaned_params_str, flags=re.IGNORECASE)
	parsed = parsed._replace(params=cleaned_params_str)
	
	# Process query string
	if parsed.query:
		params = parse_qs(parsed.query, keep_blank_values=True)
		cleaned_params = {}
		
		for key, values in params.items():
			should_strip = False
			
			for pattern in redacted_patterns:
				if re.search(pattern, key, re.IGNORECASE):
					should_strip = True
					break
				for val in values:
					if re.search(pattern, str(val), re.IGNORECASE):
						should_strip = True
						break
				if should_strip:
					break
			
			if not should_strip:
				cleaned_params[key] = values
		
		new_query = urlencode(cleaned_params, doseq=True)
		parsed = parsed._replace(query=new_query)
	
	# Clean fragment by removing redacted patterns
	cleaned_fragment = parsed.fragment
	for pattern in redacted_patterns:
		cleaned_fragment = re.sub(pattern, '', cleaned_fragment, flags=re.IGNORECASE)
	parsed = parsed._replace(fragment=cleaned_fragment)
	
	# Rebuild URL
	cleaned_url = urlunparse(parsed)
	
	unquoted_cleaned_url = unquote(cleaned_url)

	while unquoted_cleaned_url != cleaned_url:
		cleaned_url = unquoted_cleaned_url
		unquoted_cleaned_url = unquote(cleaned_url)
 	
	return cleaned_url

urls = """https://authcitizens4.com/Citizen[PRIVATE]/login/ses/index
http://noticias.bichoderua.org.br/accounts/166420/messages/83/clicks/[id]/300
http://noithatkts.com/...sys/Treace/3mail@b.c/[Recipients_group]
https://bulidhernandez.com/host[v17]/admin/js/mj.php
http://74.194.191.52/rondo.[variant].sh
http://bigger96.allow.endanger.hokoldar.ru/[Redacted]/globe/endanger/lovers.cam
http://webmail.onlinearuba.net/7b2[redacted]600a/logs.php
http://microsoftdns2.com:27688/html/jpg/U[yyyyMMddHHmmssfff].dat
http://74.194.191.52/rondo.[arch].sh]	
https://mil-by.info/#/i?id=[REDACTED]
http://source68.alternate.vadilops.ru/[Redacted]/clamp/interdependent.cbl
https://birudays.xyz/lol/excel/excel/index.php?email=redacted%40redacted.invalid"""

for url in urls.split('\n'):
  print(f'{url} \n=> {clean_url(url)}')
  print()

https://authcitizens4.com/Citizen[PRIVATE]/login/ses/index 
=> https://authcitizens4.com/Citizen/login/ses/index

http://noticias.bichoderua.org.br/accounts/166420/messages/83/clicks/[id]/300 
=> http://noticias.bichoderua.org.br/accounts/166420/messages/83/clicks/[id]/300

http://noithatkts.com/...sys/Treace/3mail@b.c/[Recipients_group] 
=> http://noithatkts.com/...sys/Treace/3mail@b.c/[Recipients_group]

https://bulidhernandez.com/host[v17]/admin/js/mj.php 
=> https://bulidhernandez.com/host[v17]/admin/js/mj.php

http://74.194.191.52/rondo.[variant].sh 
=> http://74.194.191.52/rondo.[variant].sh

http://bigger96.allow.endanger.hokoldar.ru/[Redacted]/globe/endanger/lovers.cam 
=> http://bigger96.allow.endanger.hokoldar.ru/globe/endanger/lovers.cam

http://webmail.onlinearuba.net/7b2[redacted]600a/logs.php 
=> http://webmail.onlinearuba.net/7b2600a/logs.php

http://microsoftdns2.com:27688/html/jpg/U[yyyyMMddHHmmssfff].dat 
=> http://microsoftdns2.com:27688/html/jpg/U[yyyyMMddHHmmssfff]

In [116]:
df['url'] = df['url'].progress_apply(clean_url)

100%|██████████| 2421275/2421275 [02:19<00:00, 17364.01it/s]


In [117]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True, ignore_index=True)
df.shape, df.nunique()

((1125618, 1),
 url    1125618
 dtype: int64)

In [118]:
def is_valid_url_with_scheme(url):
    """
    Check if URL has a scheme and is parseable.
    That's it - let the model learn everything else.
    """
    if not isinstance(url, str):
        return False
    
    url = url.strip()
    
    # Must have a scheme
    if '://' not in url:
        return False
    
    try:
        parsed = urlparse(url)
        # Must have both scheme and netloc to be valid
        return bool(parsed.scheme and parsed.netloc)
    except:
        return False

valid_mask = df['url'].progress_apply(is_valid_url_with_scheme)

100%|██████████| 1125618/1125618 [00:10<00:00, 105245.98it/s]


In [119]:
valid_mask.sum(), df.shape[0]

(np.int64(1125618), 1125618)

In [131]:
df

,url
0,http://194.36.191.186/vycvebnbesre4q/uqcwvebnl...
1,http://ncdnncjnrjmainmain.us/new/new/home/defa...
2,http://92.63.197.153/www/2.exe
3,https://support.ms-live.us/update/02583235891M...
4,https://mairie-fimela.com/UE.php
...,...
1125613,http://asl-company.ru/Notification-de-facture-...
1125614,http://xn--90abegbttpjb3bzb2j.xn--p1ai/doc/En/...
1125615,http://kuzina-teatr.ru/newsletter/US_us/FILE/I...
1125616,http://robertrowe.com/DOC/Past-Due-invoice


In [ ]:
# df.drop_duplicates(inplace=True)

# df.sample(frac=1, ignore_index=True).to_csv('../../datasets/malicious.csv')